In [7]:
# Cell 1: Clean AdaCLIP Setup
print("🚀 AdaCLIP Clean Implementation")

import os
import sys
import torch
import numpy as np
import random
from pathlib import Path
import json
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm import tqdm

# Paths
ADACLIP_REPO = "/scratch/leuven/369/vsc36963/Vakantiejob/Decospan/D04_Model_AdaCLIP/AdaCLIP"
DATASET_PATH = "/scratch/leuven/369/vsc36963/Vakantiejob/Decospan/Dataset"
WEIGHTS_PATH = f"{ADACLIP_REPO}/weights/pretrained_all.pth"

# Setup
os.chdir(ADACLIP_REPO)
sys.path.insert(0, ADACLIP_REPO)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f"Device: {device}")
print(f"Weights exist: {os.path.exists(WEIGHTS_PATH)}")
print(f"Dataset exists: {os.path.exists(DATASET_PATH)}")
print("✅ Setup complete")

🚀 AdaCLIP Clean Implementation
Device: cuda
Weights exist: True
Dataset exists: True
✅ Setup complete


In [8]:
# Cell 2: Dataset Loading & 80/20 Split
print("📊 Loading wood defect dataset...")

# Load dataset paths
normal_path = Path(DATASET_PATH) / "normal"
anomaly_path = Path(DATASET_PATH) / "anomaly"

# Get all normal images
normal_images = list(normal_path.glob("*.jpg"))

# Get all anomaly images with types
anomaly_images = []
for defect_folder in anomaly_path.iterdir():
    if defect_folder.is_dir():
        for img_path in defect_folder.glob("*.jpg"):
            anomaly_images.append((img_path, defect_folder.name))

# Shuffle for random split
random.shuffle(normal_images)
random.shuffle(anomaly_images)

# 80/20 split
split_idx_normal = int(len(normal_images) * 0.8)
split_idx_anomaly = int(len(anomaly_images) * 0.8)

eval_normal = normal_images[:split_idx_normal]
eval_anomaly = anomaly_images[:split_idx_anomaly]

print(f"Evaluation set: {len(eval_normal)} normal + {len(eval_anomaly)} anomaly")
print(f"Reserved: {len(normal_images) - len(eval_normal)} normal + {len(anomaly_images) - len(eval_anomaly)} anomaly")

# Combine for evaluation
eval_data = [(img, 0, "normal") for img in eval_normal] + [(img, 1, defect_type) for img, defect_type in eval_anomaly]
random.shuffle(eval_data)

print(f"Total evaluation samples: {len(eval_data)}")
print("✅ Dataset ready")

📊 Loading wood defect dataset...
Evaluation set: 7671 normal + 796 anomaly
Reserved: 1918 normal + 200 anomaly
Total evaluation samples: 8467
✅ Dataset ready


In [9]:
# Cell 3: Load Pre-trained AdaCLIP Model (Fixed prompting config)
print("🧠 Loading pre-trained AdaCLIP...")

# Set cache directory to scratch
CACHE_DIR = "/scratch/leuven/369/vsc36963/Vakantiejob/Decospan/D04_Model_AdaCLIP/cache"
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ['TORCH_HOME'] = CACHE_DIR

# Import AdaCLIP components
from method.adaclip import AdaCLIP
from method.trainer import AdaCLIP_Trainer

# ViT-L/14 configuration
model_configs = {
    "embed_dim": 768,
    "vision_cfg": {"layers": 24, "width": 1024},
    "text_cfg": {"layers": 12, "width": 768}
}

# Feature hierarchy for ViT-L/14
n_layers = model_configs['vision_cfg']['layers']
substage = n_layers // 4
features_list = [substage, substage * 2, substage * 3, substage * 4]

# Initialize AdaCLIP with CORRECT prompting config to match pretrained weights
model = AdaCLIP_Trainer(
    backbone='ViT-L/14',
    feat_list=features_list,
    input_dim=model_configs['vision_cfg']['width'],
    output_dim=model_configs['embed_dim'],
    learning_rate=1e-4,
    device=device,
    image_size=518,
    prompting_depth=4,
    prompting_length=5,  # This should match the checkpoint (5 prompts)
    prompting_type='SD',
    prompting_branch='VL',
    use_hsf=True,
    k_clusters=20
)

# Load pre-trained weights
checkpoint = torch.load(WEIGHTS_PATH, map_location=device)
model.load_state_dict(checkpoint, strict=False)
model.eval()

# Set prompts
normal_prompt = "a photo of wood with natural grain texture and black background"
anomaly_prompt = "a photo of wood with small defective texture"

print(f"Model loaded: ViT-L/14 with {features_list}")
print(f"Prompting length: 5 (matches checkpoint)")
print("✅ Model ready")

🧠 Loading pre-trained AdaCLIP...
**************************************************
Use pretrained model from openai...
**************************************************
Model loaded: ViT-L/14 with [6, 12, 18, 24]
Prompting length: 5 (matches checkpoint)
✅ Model ready


In [10]:
# Cell 4: Zero-Shot Evaluation
print("🎯 Running zero-shot evaluation...")

def evaluate_sample(model, image_path, normal_prompt, anomaly_prompt):
    """Evaluate single image with AdaCLIP"""
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = model.preprocess(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Get scores for both prompts
        normal_result = model.clip_model(image_tensor, [normal_prompt], aggregation=True)
        anomaly_result = model.clip_model(image_tensor, [anomaly_prompt], aggregation=True)
        
        # Extract scores
        _, normal_score = normal_result[:2]
        _, anomaly_score = anomaly_result[:2]
        
        # Calculate combined score (higher = more anomalous)
        combined_score = normal_score.item() - anomaly_score.item()
        
    return combined_score

# Run evaluation
results = {
    'scores': [],
    'labels': [],
    'defect_types': [],
    'image_paths': []
}

print(f"Evaluating {len(eval_data)} samples...")

for i, (image_path, label, defect_type) in enumerate(tqdm(eval_data)):
    try:
        score = evaluate_sample(model, image_path, normal_prompt, anomaly_prompt)
        
        results['scores'].append(score)
        results['labels'].append(label)
        results['defect_types'].append(defect_type)
        results['image_paths'].append(str(image_path))
        
    except Exception as e:
        print(f"Failed on image {i}: {e}")
        continue

# Calculate metrics
scores = np.array(results['scores'])
labels = np.array(results['labels'])

auroc = roc_auc_score(labels, scores)
ap = average_precision_score(labels, scores)

# Results summary
normal_count = np.sum(labels == 0)
anomaly_count = np.sum(labels == 1)

print(f"\n📊 Results:")
print(f"Samples evaluated: {len(scores)}")
print(f"Normal: {normal_count}, Anomaly: {anomaly_count}")
print(f"AUROC: {auroc:.4f}")
print(f"Average Precision: {ap:.4f}")
print("✅ Evaluation complete")

🎯 Running zero-shot evaluation...
Evaluating 8467 samples...


100%|██████████| 8467/8467 [27:01<00:00,  5.22it/s]


📊 Results:
Samples evaluated: 8467
Normal: 7671, Anomaly: 796
AUROC: 0.8944
Average Precision: 0.6084
✅ Evaluation complete


In [11]:
# CLEAN AdaCLIP Visualization - No Duplicates
print("📊 Creating clean AdaCLIP visualizations...")

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import shutil
# =============================================================================
# RECREATE ALL INDICES (to be sure they exist and are correct)
# =============================================================================
# =============================================================================
# COMPLETE MISSING VARIABLES FIX
# =============================================================================
import cv2
import numpy as np
from scipy.ndimage import gaussian_filter
from sklearn.metrics import roc_curve, confusion_matrix
import shutil

print("🔧 Recreating ALL missing variables...")

# Calculate predictions and metrics
fpr, tpr, thresholds = roc_curve(labels, scores)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
predictions = (scores >= optimal_threshold).astype(int)

cm = confusion_matrix(labels, predictions)
accuracy = (cm[0,0] + cm[1,1]) / cm.sum()
precision = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1] + cm[0,1]) > 0 else 0
recall = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1] + cm[1,0]) > 0 else 0

# Recreate false_positives and false_negatives lists
false_positives = []
false_negatives = []

for i, (score, label, defect_type, img_path) in enumerate(zip(scores, labels, results['defect_types'], results['image_paths'])):
    pred = predictions[i]
    
    if label == 0 and pred == 1:  # False Positive
        false_positives.append({
            'index': i,
            'path': img_path,
            'score': score,
            'defect_type': defect_type,
            'true_label': 'normal',
            'predicted': 'anomaly'
        })
    elif label == 1 and pred == 0:  # False Negative
        false_negatives.append({
            'index': i,
            'path': img_path,
            'score': score,
            'defect_type': defect_type,
            'true_label': 'anomaly', 
            'predicted': 'normal'
        })

# Sort lists
false_positives.sort(key=lambda x: x['score'], reverse=True)
false_negatives.sort(key=lambda x: x['score'])

print(f"✅ Recreated ALL variables:")
print(f"   • Accuracy: {accuracy:.4f}")
print(f"   • False Positives: {len(false_positives)}")
print(f"   • False Negatives: {len(false_negatives)}")
def get_adaclip_heatmap_working(model, image_path, text_prompt):
    """Get heatmap from AdaCLIP using the official method from app.py"""
    try:
        from scipy.ndimage import gaussian_filter
        
        # Load and preprocess image
        image = Image.open(image_path).convert('RGB')
        np_image = np.array(image)
        np_image = cv2.cvtColor(np_image, cv2.COLOR_RGB2BGR)
        np_image = cv2.resize(np_image, (518, 518))
        
        # Preprocess for model
        img_input = model.preprocess(image).unsqueeze(0)
        img_input = img_input.to(model.device)
        
        with torch.no_grad():
            # Key: use aggregation=True and take FIRST output
            anomaly_map, anomaly_score = model.clip_model(img_input, [text_prompt], aggregation=True)
        
        # Process anomaly map (following official app.py exactly)
        anomaly_map = anomaly_map[0, :, :].cpu().numpy()
        anomaly_score = anomaly_score[0].cpu().numpy()
        anomaly_map = gaussian_filter(anomaly_map, sigma=4)
        anomaly_map = (anomaly_map * 255).astype(np.uint8)
        
        # Resize to original image size
        orig_size = image.size  # (W, H)
        if anomaly_map.shape != (orig_size[1], orig_size[0]):
            anomaly_map = cv2.resize(anomaly_map, orig_size, interpolation=cv2.INTER_CUBIC)
        
        return anomaly_map, anomaly_score
        
    except Exception as e:
        print(f"Heatmap generation failed: {e}")
        return None, None

def create_heatmap_overlay_global_scale(image, heatmap, global_min, global_max, alpha=0.6):
    """Create heatmap overlay with GLOBAL color scale"""
    if heatmap is None:
        return image
    
    # Convert image to numpy if needed
    if isinstance(image, Image.Image):
        np_image = np.array(image)
    else:
        np_image = (image * 255).astype(np.uint8) if image.max() <= 1 else image.astype(np.uint8)
    
    # GLOBAL normalization - same scale for ALL heatmaps
    if global_max > global_min:
        heatmap_norm = np.clip((heatmap - global_min) / (global_max - global_min), 0, 1)
    else:
        heatmap_norm = np.ones_like(heatmap) * 0.5
    
    heatmap_norm = (heatmap_norm * 255).astype(np.uint8)
    
    # Use consistent colormap
    heat_map = cv2.applyColorMap(heatmap_norm, cv2.COLORMAP_JET)
    
    # Convert RGB to BGR for OpenCV
    if len(np_image.shape) == 3:
        np_image_bgr = cv2.cvtColor(np_image, cv2.COLOR_RGB2BGR)
    else:
        np_image_bgr = np_image
    
    # Blend
    vis_map = cv2.addWeighted(heat_map, alpha, np_image_bgr, 1-alpha, 0)
    
    # Convert back to RGB
    vis_map_rgb = cv2.cvtColor(vis_map, cv2.COLOR_BGR2RGB)
    
    return vis_map_rgb
# Calculate predictions with optimal threshold
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(labels, scores)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
predictions = (scores >= optimal_threshold).astype(int)

# Create all category indices
correct_normal_idx = [i for i in range(len(scores)) if labels[i] == 0 and predictions[i] == 0]  # TN
correct_anomaly_idx = [i for i in range(len(scores)) if labels[i] == 1 and predictions[i] == 1]  # TP
false_positive_idx = [i for i in range(len(scores)) if labels[i] == 0 and predictions[i] == 1]  # FP
false_negative_idx = [i for i in range(len(scores)) if labels[i] == 1 and predictions[i] == 0]  # FN

print(f"✅ Recreated indices:")
print(f"   • True Negatives (TN): {len(correct_normal_idx)}")
print(f"   • True Positives (TP): {len(correct_anomaly_idx)}")
print(f"   • False Positives (FP): {len(false_positive_idx)}")
print(f"   • False Negatives (FN): {len(false_negative_idx)}")
print(f"   • Total: {len(correct_normal_idx) + len(correct_anomaly_idx) + len(false_positive_idx) + len(false_negative_idx)}")
# =============================================================================
# 1. PROMPTS (EASILY CHANGEABLE)
# =============================================================================

print(f"🎯 Using prompts:")
print(f"   • Normal: '{normal_prompt}'")
print(f"   • Anomaly: '{anomaly_prompt}'")

# =============================================================================
# 2. CALCULATE GLOBAL SCALE (ACTUAL MIN/MAX)
# =============================================================================
def calculate_global_scale():
    """Calculate ACTUAL min/max across all categories"""
    print("📊 Calculating global heatmap scale...")
    
    all_heatmaps = []
    categories = [
        ("TN", correct_normal_idx[:150]),
        ("TP", correct_anomaly_idx[:150]),
        ("FP", false_positive_idx[:min(150, len(false_positive_idx))]),  # Max beschikbare
        ("FN", false_negative_idx[:min(150, len(false_negative_idx))])   # Max beschikbare
    ]
    
    for cat_name, indices in categories:
        if len(indices) > 0:
            print(f"   Processing {cat_name}: {len(indices)} samples")
            for idx in indices:
                img_path = results['image_paths'][idx]
                heatmap, _ = get_adaclip_heatmap_working(model, img_path, anomaly_prompt)
                if heatmap is not None:
                    all_heatmaps.append(heatmap)
    
    if len(all_heatmaps) > 0:
        all_values = np.concatenate([h.flatten() for h in all_heatmaps])
        global_min = float(all_values.min())
        global_max = float(all_values.max())
        
        print(f"   ✅ Global range: {global_min:.1f} to {global_max:.1f}")
        print(f"   📊 Range width: {global_max - global_min:.1f}")
        return global_min, global_max
    else:
        return 0, 255

GLOBAL_MIN, GLOBAL_MAX = calculate_global_scale()

# =============================================================================
# 3. CREATE RESULTS FOLDER
# =============================================================================
results_folder = Path("/scratch/leuven/369/vsc36963/Vakantiejob/Decospan/D04_Model_AdaCLIP/results")
results_folder.mkdir(exist_ok=True)
print(f"📁 Results will be saved to: {results_folder}")

# =============================================================================
# 4. PLOT FUNCTION (ONE CLEAN VERSION)
# =============================================================================
def plot_category(category_name, short_name, indices, color):
    """Plot 4x4 grid for one category"""
    print(f"   Creating {short_name}...")
    
    # Select 16 samples
    if len(indices) >= 16:
        selected_idx = np.random.choice(indices, 16, replace=False)
    else:
        selected_idx = indices[:16] if len(indices) > 0 else []
    
    # Create figure
    fig, axes = plt.subplots(4, 8, figsize=(20, 10))
    fig.suptitle(f'AdaCLIP {short_name} - {category_name} (Range: {GLOBAL_MIN:.1f}-{GLOBAL_MAX:.1f})', 
                fontsize=14, fontweight='bold', color=color, y=0.95)
    
    # Plot samples
    for row in range(4):
        for col in range(4):
            sample_num = row * 4 + col
            ax_img = axes[row, col * 2]
            ax_heat = axes[row, col * 2 + 1]
            
            if sample_num < len(selected_idx):
                idx = selected_idx[sample_num]
                img_path = results['image_paths'][idx]
                score = scores[idx]
                defect_type = results['defect_types'][idx]
                
                # Get heatmap
                heatmap, _ = get_adaclip_heatmap_working(model, img_path, anomaly_prompt)
                
                # Load and show image
                image = Image.open(img_path).convert('RGB')
                img_array = np.array(image)
                ax_img.imshow(img_array)
                
                filename = Path(img_path).name
                if len(filename) > 12:
                    filename = filename[:9] + "..."
                ax_img.set_title(f'{filename}\n{defect_type}\nScore: {score:.2f}', 
                                fontsize=8, color=color, pad=5)
                ax_img.axis('off')
                
                # Show heatmap with global scale
                if heatmap is not None:
                    overlay = create_heatmap_overlay_global_scale(
                        image, heatmap, GLOBAL_MIN, GLOBAL_MAX, alpha=0.7
                    )
                    ax_heat.imshow(overlay)
                    local_min = float(heatmap.min())
                    local_max = float(heatmap.max())
                    ax_heat.set_title(f'{local_min:.0f}-{local_max:.0f}', 
                                     fontsize=8, color=color, pad=5)
                else:
                    ax_heat.text(0.5, 0.5, 'No heatmap', ha='center', va='center')
                ax_heat.axis('off')
            else:
                ax_img.axis('off')
                ax_heat.axis('off')
    
    # Add colorbar
    norm = Normalize(vmin=GLOBAL_MIN, vmax=GLOBAL_MAX)
    sm = ScalarMappable(norm=norm, cmap='jet')
    sm.set_array([])
    cbar_ax = fig.add_axes([0.96, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label('Global Anomaly Score', rotation=270, labelpad=15)
    
    plt.subplots_adjust(top=0.9, bottom=0.05, left=0.05, right=0.94, hspace=0.3, wspace=0.1)
    
    # Save
    save_path = results_folder / f'adaclip_{short_name.lower()}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# =============================================================================
# 5. COMPARISON PLOT FUNCTION
# =============================================================================
def create_comparison():
    """Create comparison of all categories"""
    print("   Creating comparison plot...")
    
    categories = [
        ("True Negatives", "TN", correct_normal_idx[:4], 'green'),
        ("True Positives", "TP", correct_anomaly_idx[:4], 'blue'),
        ("False Positives", "FP", false_positive_idx[:4], 'red'),
        ("False Negatives", "FN", false_negative_idx[:4], 'orange')
    ]
    
    fig, axes = plt.subplots(4, 8, figsize=(20, 12))
    fig.suptitle(f'AdaCLIP Comparison All Categories (Range: {GLOBAL_MIN:.1f}-{GLOBAL_MAX:.1f})', 
                fontsize=16, fontweight='bold', y=0.95)
    
    for row, (cat_name, short_name, indices, color) in enumerate(categories):
        # Add category label
        fig.text(0.02, 0.85 - row * 0.2, short_name, fontsize=16, fontweight='bold', 
                color=color, rotation=0, ha='center', va='center')
        
        for col in range(4):
            ax_img = axes[row, col * 2]
            ax_heat = axes[row, col * 2 + 1]
            
            if col < len(indices):
                idx = indices[col]
                img_path = results['image_paths'][idx]
                score = scores[idx]
                heatmap, _ = get_adaclip_heatmap_working(model, img_path, anomaly_prompt)
                
                # Show image
                image = Image.open(img_path).convert('RGB')
                img_array = np.array(image)
                ax_img.imshow(img_array)
                
                filename = Path(img_path).name
                if len(filename) > 12:
                    filename = filename[:9] + "..."
                ax_img.set_title(f'{filename}\nScore: {score:.2f}', 
                                fontsize=8, color=color, pad=5)
                ax_img.axis('off')
                
                # Show heatmap
                if heatmap is not None:
                    overlay = create_heatmap_overlay_global_scale(
                        image, heatmap, GLOBAL_MIN, GLOBAL_MAX, alpha=0.7
                    )
                    ax_heat.imshow(overlay)
                    local_min = float(heatmap.min())
                    local_max = float(heatmap.max())
                    ax_heat.set_title(f'{local_min:.0f}-{local_max:.0f}', 
                                     fontsize=8, color=color, pad=5)
                else:
                    ax_heat.text(0.5, 0.5, 'No heatmap', ha='center', va='center')
                ax_heat.axis('off')
            else:
                ax_img.axis('off')
                ax_heat.axis('off')
    
    # Add colorbar
    norm = Normalize(vmin=GLOBAL_MIN, vmax=GLOBAL_MAX)
    sm = ScalarMappable(norm=norm, cmap='jet')
    sm.set_array([])
    cbar_ax = fig.add_axes([0.96, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label('Global Anomaly Score', rotation=270, labelpad=15)
    
    plt.subplots_adjust(top=0.9, bottom=0.05, left=0.08, right=0.94, hspace=0.3, wspace=0.1)
    
    save_path = results_folder / 'adaclip_comparison_all_categories.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
# =============================================================================
# CREATE PERFORMANCE ANALYSIS PLOT
# =============================================================================
def create_performance_plot():
    """Create performance analysis with ROC, distribution, confusion matrix"""
    print("   Creating performance analysis plot...")
    
    from sklearn.metrics import roc_curve, confusion_matrix
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. ROC Curve
    fpr, tpr, thresholds = roc_curve(labels, scores)
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[optimal_idx]
    
    axes[0,0].plot(fpr, tpr, linewidth=2, label=f'AdaCLIP (AUROC = {auroc:.4f})')
    axes[0,0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axes[0,0].set_xlabel('False Positive Rate')
    axes[0,0].set_ylabel('True Positive Rate')
    axes[0,0].set_title('ROC Curve')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Score Distribution
    normal_scores = scores[labels == 0]
    anomaly_scores = scores[labels == 1]
    
    axes[0,1].hist(normal_scores, bins=50, alpha=0.7, label=f'Normal (n={len(normal_scores)})', color='green', density=True)
    axes[0,1].hist(anomaly_scores, bins=50, alpha=0.7, label=f'Anomaly (n={len(anomaly_scores)})', color='red', density=True)
    axes[0,1].axvline(optimal_threshold, color='black', linestyle='--', label=f'Threshold: {optimal_threshold:.3f}')
    axes[0,1].set_xlabel('AdaCLIP Score')
    axes[0,1].set_ylabel('Density')
    axes[0,1].set_title('Score Distribution')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Confusion Matrix
    predictions = (scores >= optimal_threshold).astype(int)
    cm = confusion_matrix(labels, predictions)
    
    im = axes[0,2].imshow(cm, interpolation='nearest', cmap='Blues')
    axes[0,2].set_title(f'Confusion Matrix\nAcc: {accuracy:.3f} | Prec: {precision:.3f} | Rec: {recall:.3f}')
    axes[0,2].set_xticks([0, 1])
    axes[0,2].set_yticks([0, 1])
    axes[0,2].set_xticklabels(['Normal', 'Anomaly'])
    axes[0,2].set_yticklabels(['Normal', 'Anomaly'])
    
    for i in range(2):
        for j in range(2):
            axes[0,2].text(j, i, f'{cm[i, j]}', ha="center", va="center", color="black", fontsize=14)
    
    # 4. Per-Defect Performance
    defect_performance = {}
    for defect_type in set(results['defect_types']):
        if defect_type == 'normal':
            continue
        defect_indices = [i for i, dt in enumerate(results['defect_types']) if dt == defect_type]
        if len(defect_indices) > 0:
            defect_scores = [scores[i] for i in defect_indices]
            defect_labels = [labels[i] for i in defect_indices]
            defect_preds = [predictions[i] for i in defect_indices]
            
            correct = sum(1 for label, pred in zip(defect_labels, defect_preds) if label == pred)
            accuracy_def = correct / len(defect_labels)
            defect_performance[defect_type] = {'accuracy': accuracy_def, 'count': len(defect_indices)}
    
    if defect_performance:
        sorted_defects = sorted(defect_performance.items(), key=lambda x: x[1]['count'], reverse=True)[:10]
        defect_names = [item[0] for item in sorted_defects]
        defect_accuracies = [item[1]['accuracy'] for item in sorted_defects]
        defect_counts = [item[1]['count'] for item in sorted_defects]
        
        bars = axes[1,0].bar(range(len(defect_names)), defect_accuracies, color='skyblue')
        axes[1,0].set_xticks(range(len(defect_names)))
        axes[1,0].set_xticklabels(defect_names, rotation=45, ha='right')
        axes[1,0].set_ylabel('Accuracy')
        axes[1,0].set_title('Per-Defect Performance')
        axes[1,0].grid(True, alpha=0.3)
        
        for i, (bar, count) in enumerate(zip(bars, defect_counts)):
            axes[1,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                          f'n={count}', ha='center', va='bottom', fontsize=8)
    
    # 5. Performance Summary
    axes[1,1].axis('off')
    axes[1,1].text(0.1, 0.9, "Performance Summary", transform=axes[1,1].transAxes, fontsize=14, fontweight='bold')
    axes[1,1].text(0.1, 0.7, f"AUROC: {auroc:.4f}", transform=axes[1,1].transAxes, fontsize=12)
    axes[1,1].text(0.1, 0.6, f"Accuracy: {accuracy:.4f}", transform=axes[1,1].transAxes, fontsize=12)
    axes[1,1].text(0.1, 0.5, f"Precision: {precision:.4f}", transform=axes[1,1].transAxes, fontsize=12)
    axes[1,1].text(0.1, 0.4, f"Recall: {recall:.4f}", transform=axes[1,1].transAxes, fontsize=12)
    
    # 6. Score Statistics
    axes[1,2].axis('off')
    axes[1,2].text(0.1, 0.9, "Score Statistics", transform=axes[1,2].transAxes, fontsize=14, fontweight='bold')
    axes[1,2].text(0.1, 0.7, f"Global Min: {GLOBAL_MIN:.1f}", transform=axes[1,2].transAxes, fontsize=12)
    axes[1,2].text(0.1, 0.6, f"Global Max: {GLOBAL_MAX:.1f}", transform=axes[1,2].transAxes, fontsize=12)
    axes[1,2].text(0.1, 0.5, f"Range: {GLOBAL_MAX - GLOBAL_MIN:.1f}", transform=axes[1,2].transAxes, fontsize=12)
    
    plt.tight_layout()
    save_path = results_folder / 'adaclip_performance_analysis.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()


# =============================================================================
# 6. CREATE ERROR ANALYSIS FOLDERS
# =============================================================================
def create_error_folders():
    """Create folders with copied images"""
    print("📁 Creating error analysis folders...")
    
    # Create folders
    fp_folder = results_folder / "false_anomalies_FP"
    fn_folder = results_folder / "false_normals_FN"
    fp_folder.mkdir(exist_ok=True)
    fn_folder.mkdir(exist_ok=True)
    
    # Copy FP images and create analysis
    fp_file = fp_folder / "analysis.txt"
    with open(fp_file, 'w') as f:
        f.write(f"=== FALSE POSITIVES Analysis ===\n\n")
        f.write(f"Total: {len(false_positives)}\n")
        f.write(f"AUROC: {auroc:.4f}\n")
        f.write(f"Global range: {GLOBAL_MIN:.1f} to {GLOBAL_MAX:.1f}\n\n")
        
        for i, fp in enumerate(false_positives):
            source_path = Path(fp['path'])
            dest_path = fp_folder / source_path.name
            try:
                shutil.copy2(source_path, dest_path)
                f.write(f"{i+1:3d}. Score: {fp['score']:7.4f} | Image: {source_path.name}\n")
            except:
                f.write(f"{i+1:3d}. Score: {fp['score']:7.4f} | COPY FAILED | {source_path.name}\n")
    
    # Copy FN images and create analysis
    fn_file = fn_folder / "analysis.txt"
    with open(fn_file, 'w') as f:
        f.write(f"=== FALSE NEGATIVES Analysis ===\n\n")
        f.write(f"Total: {len(false_negatives)}\n")
        f.write(f"AUROC: {auroc:.4f}\n")
        f.write(f"Global range: {GLOBAL_MIN:.1f} to {GLOBAL_MAX:.1f}\n\n")
        
        for i, fn in enumerate(false_negatives):
            source_path = Path(fn['path'])
            dest_path = fn_folder / source_path.name
            try:
                shutil.copy2(source_path, dest_path)
                f.write(f"{i+1:3d}. Score: {fn['score']:7.4f} | Type: {fn['defect_type']:15s} | Image: {source_path.name}\n")
            except:
                f.write(f"{i+1:3d}. Score: {fn['score']:7.4f} | Type: {fn['defect_type']:15s} | COPY FAILED | {source_path.name}\n")
    
    print(f"   ✅ FP folder: {fp_folder} ({len(false_positives)} images)")
    print(f"   ✅ FN folder: {fn_folder} ({len(false_negatives)} images)")

# =============================================================================
# 7. CREATE ALL VISUALIZATIONS (ONCE!)
# =============================================================================
print(f"\n🎯 Creating visualizations with global scale: {GLOBAL_MIN:.1f} to {GLOBAL_MAX:.1f}")
# Create performance analysis plot
create_performance_plot()
# Create error folders
create_error_folders()

# Create comparison plot
create_comparison()

# Create individual category plots
print("📊 Creating individual category plots...")
if len(correct_normal_idx) > 0:
    plot_category("True Negatives", "TN", correct_normal_idx, 'green')

if len(correct_anomaly_idx) > 0:
    plot_category("True Positives", "TP", correct_anomaly_idx, 'blue')

if len(false_positive_idx) > 0:
    plot_category("False Positives", "FP", false_positive_idx, 'red')

if len(false_negative_idx) > 0:
    plot_category("False Negatives", "FN", false_negative_idx, 'orange')

# =============================================================================
# 8. FINAL SUMMARY (ONCE!)
# =============================================================================
print(f"\n✅ AdaCLIP analysis complete!")
print(f"🎯 AUROC: {auroc:.4f}")
print(f"📊 Global range: {GLOBAL_MIN:.1f} to {GLOBAL_MAX:.1f}")
print(f"📁 All results saved in: {results_folder}")

print(f"\n📋 Created files:")
print(f"   🖼️ adaclip_tn.png")
print(f"   🖼️ adaclip_tp.png") 
print(f"   🖼️ adaclip_fp.png")
print(f"   🖼️ adaclip_fn.png")
print(f"   🖼️ adaclip_comparison_all_categories.png")
print(f"   📁 false_anomalies_FP/ ({len(false_positives)} images)")
print(f"   📁 false_normals_FN/ ({len(false_negatives)} images)")

print(f"\n🔧 To test different prompts:")
print(f"   1. Change normal_prompt and anomaly_prompt at the top")
print(f"   2. Re-run this cell")
print(f"   3. Compare AUROC results!")

📊 Creating clean AdaCLIP visualizations...
🔧 Recreating ALL missing variables...
✅ Recreated ALL variables:
   • Accuracy: 0.8795
   • False Positives: 856
   • False Negatives: 164
✅ Recreated indices:
   • True Negatives (TN): 6815
   • True Positives (TP): 632
   • False Positives (FP): 856
   • False Negatives (FN): 164
   • Total: 8467
🎯 Using prompts:
   • Normal: 'a photo of wood with natural grain texture and black background'
   • Anomaly: 'a photo of wood with small defective texture'
📊 Calculating global heatmap scale...
   Processing TN: 150 samples
   Processing TP: 150 samples
   Processing FP: 150 samples
   Processing FN: 150 samples
   ✅ Global range: 30.0 to 181.0
   📊 Range width: 151.0
📁 Results will be saved to: /scratch/leuven/369/vsc36963/Vakantiejob/Decospan/D04_Model_AdaCLIP/results

🎯 Creating visualizations with global scale: 30.0 to 181.0
   Creating performance analysis plot...
📁 Creating error analysis folders...
   ✅ FP folder: /scratch/leuven/369/vsc3696